# EDA — Track Stats

Exploratory analysis of the album track-statistics feature block. This notebook answers five questions that motivated key decisions in the track stats pipeline:

1. **Album coverage** — how many albums have track-length data at all, and how complete is it?
2. **Track length distribution** — what does a typical album look like, and where are the outliers?
3. **Track count distribution** — single-track vs EP vs full album vs box set?
4. **First release year** — imputed vs true coverage, and how the imputation shifts the distribution
5. **Feature correlations** — which columns are redundant (guiding the exclusion list in the feature notebook)

**Input:** `../data/sql_feature_album_track_stats.parquet`

**Run after:** `1-data/04-feature-track-stats-import.ipynb`. Display-only — writes nothing to `data/`.

## Setup

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

FEATURES_DIR = '../data/features'
DATA_DIR     = '../data'
MS_TO_SEC    = 1_000
MS_TO_MIN    = 60_000

In [ ]:
df = pd.read_parquet(f'{DATA_DIR}/sql_feature_album_track_stats.parquet')

with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)
album_index = pd.Index(album_ids)
n_albums = len(album_index)

# Align to master album universe
df = df.rename(columns={'release_group_id': 'album_id'})
df = df[df['album_id'].isin(album_index)].copy()

# Add human-scale columns used throughout
df['total_length_min']  = df['total_length_ms']  / MS_TO_MIN
df['mean_length_sec']   = df['mean_length_ms']   / MS_TO_SEC
df['median_length_sec'] = df['median_length_ms'] / MS_TO_SEC
df['min_length_sec']    = df['min_length_ms']    / MS_TO_SEC
df['max_length_sec']    = df['max_length_ms']    / MS_TO_SEC

print(f'Master album universe : {n_albums:,}')
print(f'Track-stats rows      : {len(df):,}')
print(f'Columns               : {df.columns.tolist()}')

---
# Part 1 — Album Coverage

The parquet has one row per release group (album). Not every album in the universe has track-length
data — some are data-only releases, some have no track durations entered in MusicBrainz. This
section measures how many albums have usable data and how complete the length coverage is within
each album.

In [ ]:
has_any_row       = len(df)
has_length        = df['total_length_ms'].notna().sum()
has_year          = df['first_release_year'].notna().sum()
has_imputed_year  = df['first_release_year_imputed'].notna().sum()
fully_complete    = df[df['pct_tracks_with_length'] == 100].shape[0]
partially_missing = df[(df['pct_tracks_with_length'] > 0) & (df['pct_tracks_with_length'] < 100)].shape[0]
no_lengths        = df[df['pct_tracks_with_length'] == 0].shape[0]

print(f'Albums in master universe          : {n_albums:,}')
print(f'Albums with a track-stats row      : {has_any_row:,}  ({has_any_row/n_albums*100:.1f}%)')
print()
print('Length completeness (within albums that have a row):')
print(f'  100% of tracks have length       : {fully_complete:,}  ({fully_complete/has_any_row*100:.1f}%)')
print(f'  Some tracks missing length       : {partially_missing:,}  ({partially_missing/has_any_row*100:.1f}%)')
print(f'  0 tracks have length             : {no_lengths:,}  ({no_lengths/has_any_row*100:.1f}%)')
print()
print('Year coverage:')
print(f'  Albums with true first_release_year     : {has_year:,}  ({has_year/has_any_row*100:.1f}%)')
print(f'  Albums with imputed first_release_year  : {has_imputed_year:,}  ({has_imputed_year/has_any_row*100:.1f}%)')

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pct = df['pct_tracks_with_length']
sns.histplot(pct, bins=20, ax=axes[0], color='#4A90E2')
axes[0].set_title('% of Tracks with Length Data per Album', weight='bold')
axes[0].set_xlabel('% of Tracks with Length')
axes[0].set_ylabel('Count of Albums')

# CDF
sorted_pct = np.sort(pct.dropna())
cdf = np.arange(1, len(sorted_pct) + 1) / len(sorted_pct)
axes[1].plot(sorted_pct, cdf, color='#4A90E2', linewidth=2)
axes[1].axvline(80, color='red', linestyle='--', linewidth=1, label='80% threshold')
axes[1].set_title('CDF — % Tracks with Length', weight='bold')
axes[1].set_xlabel('% of Tracks with Length')
axes[1].set_ylabel('Cumulative Fraction of Albums')
axes[1].legend()

plt.tight_layout()
plt.show()

### Most albums that have a row are either fully complete or fully empty

The distribution is bimodal: albums cluster at 100% coverage or 0%. The partially-complete middle
is a small fraction. Albums at 0% carry track count and year information but no length statistics —
the feature notebook imputes length columns with column medians before scaling.

---
# Part 2 — Track Length Distribution

Track length (per-track mean and median) is the densest signal in this feature block. This section
examines the distribution and identifies the outlier regimes: very short releases (interludes,
noise records) and very long ones (ambient, spoken word, box sets).

In [ ]:
# Filter to albums with length data for this section
df_len = df[df['mean_length_ms'].notna()].copy()

mean_sec = df_len['mean_length_sec']
p1, p99  = mean_sec.quantile(0.01), mean_sec.quantile(0.99)

print(f'Albums with mean track length data: {len(df_len):,}')
print()
print('Mean track length distribution (seconds):')
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    v = mean_sec.quantile(p / 100)
    print(f'  p{p:<3}: {v:>8.1f}s  ({v/60:.1f} min)')
print()
print(f'Albums with mean track < 60s  (very short) : {(mean_sec < 60).sum():>8,}  ({(mean_sec < 60).mean()*100:.1f}%)')
print(f'Albums with mean track > 900s (15+ min avg): {(mean_sec > 900).sum():>8,}  ({(mean_sec > 900).mean()*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Clipped to p1–p99 for readability
clipped = mean_sec.clip(p1, p99)
sns.histplot(clipped / 60, bins=60, ax=axes[0], color='#27AE60')
axes[0].axvline(3.5, color='red',    linestyle='--', linewidth=1, label='3.5 min')
axes[0].axvline(5.5, color='orange', linestyle='--', linewidth=1, label='5.5 min')
axes[0].set_title('Mean Track Length per Album (p1–p99 clipped)', weight='bold')
axes[0].set_xlabel('Mean Track Length (minutes)')
axes[0].set_ylabel('Count of Albums')
axes[0].legend()

# Total album length
total_min = df_len['total_length_min']
tp1, tp99 = total_min.quantile(0.01), total_min.quantile(0.99)
clipped_total = total_min.clip(tp1, tp99)
sns.histplot(clipped_total, bins=60, ax=axes[1], color='#E67E22')
axes[1].axvline(35,  color='red',    linestyle='--', linewidth=1, label='35 min')
axes[1].axvline(80,  color='orange', linestyle='--', linewidth=1, label='80 min')
axes[1].set_title('Total Album Length (p1–p99 clipped)', weight='bold')
axes[1].set_xlabel('Total Length (minutes)')
axes[1].set_ylabel('Count of Albums')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print('=== Very short releases (mean track < 60s) ===')
short = df_len[df_len['mean_length_sec'] < 60][['album_id', 'track_count', 'mean_length_sec', 'total_length_min']]
print(short.describe().round(1).to_string())
print()
print('=== Very long releases (mean track > 20 min) ===')
long = df_len[df_len['mean_length_sec'] > 1200][['album_id', 'track_count', 'mean_length_sec', 'total_length_min']]
print(long.describe().round(1).to_string())

### Length centres on the 3–5 minute pop/rock norm, with meaningful outlier tails

- The bulk of albums cluster around a 3–5 minute mean — standard song length across most genres.
- Very short releases (mean < 60s) are mostly noise records, interludes, and data-entry artefacts.
- Very long releases (mean > 20 min) are ambient, classical, jazz improv, or spoken-word.

Both tails are real genre signals, not errors — MinMaxScaler in the feature notebook preserves them
as extreme values rather than clipping.

---
# Part 3 — Track Count Distribution

Track count distinguishes singles and EPs from standard albums and from box sets. Understanding
the distribution confirms that the `valid_albums` scope filter is doing its job — we should not
see many single-track releases in this universe.

In [ ]:
tc = df['track_count']

print('Track count distribution:')
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f'  p{p:<3}: {tc.quantile(p/100):.0f} tracks')
print()

bands = [
    (1,  1,  '1 track (single)'),
    (2,  5,  '2–5 tracks (EP)'),
    (6,  14, '6–14 tracks (standard album)'),
    (15, 29, '15–29 tracks (long / double album)'),
    (30, None, '30+ tracks (box set / compilation)'),
]
total = len(tc)
for lo, hi, label in bands:
    if hi is None:
        n = (tc >= lo).sum()
    else:
        n = ((tc >= lo) & (tc <= hi)).sum()
    print(f'  {label:<40} {n:>9,}  ({n/total*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Clip at p99 for readability
p99_tc = tc.quantile(0.99)
clipped_tc = tc.clip(upper=p99_tc)
sns.histplot(clipped_tc, bins=50, ax=axes[0], color='#8E44AD')
axes[0].axvline(6,  color='red',    linestyle='--', linewidth=1, label='6 (EP→album)')
axes[0].axvline(15, color='orange', linestyle='--', linewidth=1, label='15 (double album)')
axes[0].set_title('Track Count Distribution (p99 clipped)', weight='bold')
axes[0].set_xlabel('Track Count')
axes[0].set_ylabel('Count of Albums')
axes[0].legend()

# Medium count
mc = df['medium_count']
mc_clipped = mc.clip(upper=mc.quantile(0.999))
sns.histplot(mc_clipped, bins=range(1, int(mc_clipped.max()) + 2), ax=axes[1], color='#16A085')
axes[1].set_title('Medium (Disc) Count Distribution', weight='bold')
axes[1].set_xlabel('Number of Discs / Mediums')
axes[1].set_ylabel('Count of Albums')
axes[1].set_xlim(0, 10)

plt.tight_layout()
plt.show()

### The album universe peaks in the 8–14 track range, with a long tail of extended releases

- The 6–14 band covers the majority — matching the typical full-length album format.
- Single-track and EP-length entries are a small residual left by the `valid_albums` filter.
  They are legitimate release-group types (albums that happen to be short) rather than filter
  failures.
- Most albums are single-disc; multi-disc releases are a small but real subset carrying a
  distinct `medium_count` signal.

> **Design decision — keep track_count and medium_count as separate features:** they are not
  redundant — a 30-track double-LP and a 30-track compilation CD have the same track count but
  different medium counts and very different length profiles.

---
# Part 4 — First Release Year

The `first_release_year` column is used by the temporal feature block as a ground-truth anchor.
This section measures true vs imputed year coverage and checks whether imputation shifts the
distribution in a way that could bias era classification.

In [ ]:
true_year    = df['first_release_year'].dropna()
imputed_year = df['first_release_year_imputed'].dropna()
has_true     = df['first_release_year'].notna()
has_imputed  = df['first_release_year_imputed'].notna()
imputed_only = has_imputed & ~has_true

print(f'Albums in this subset              : {len(df):,}')
print(f'Albums with true first_release_year: {has_true.sum():,}  ({has_true.mean()*100:.1f}%)')
print(f'Albums with imputed year           : {has_imputed.sum():,}  ({has_imputed.mean()*100:.1f}%)')
print(f'Albums imputed-only (no true year) : {imputed_only.sum():,}  ({imputed_only.mean()*100:.1f}%)')
print(f'Albums with no year at all         : {(~has_imputed).sum():,}  ({(~has_imputed).mean()*100:.1f}%)')
print()
print('True year range   :', int(true_year.min()),    '–', int(true_year.max()))
print('Imputed year range:', int(imputed_year.min()), '–', int(imputed_year.max()))

In [ ]:
# Clip obviously bad data-entry years
true_clipped    = true_year[(true_year >= 1900) & (true_year <= 2025)]
imputed_clipped = imputed_year[(imputed_year >= 1900) & (imputed_year <= 2025)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(true_clipped,    bins=range(1900, 2026), ax=axes[0], color='#2980B9', label='True year', alpha=0.7)
sns.histplot(imputed_clipped, bins=range(1900, 2026), ax=axes[0], color='#E74C3C', label='Imputed year', alpha=0.4)
axes[0].set_title('First Release Year — True vs Imputed', weight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count of Albums')
axes[0].legend()

# Imputed-only albums (gap-fill)
imputed_gap = imputed_year[imputed_only]
imputed_gap_clipped = imputed_gap[(imputed_gap >= 1900) & (imputed_gap <= 2025)]
sns.histplot(imputed_gap_clipped, bins=range(1900, 2026), ax=axes[1], color='#E74C3C')
axes[1].set_title('Imputed-Only Albums — Year Distribution', weight='bold')
axes[1].set_xlabel('Year (imputed)')
axes[1].set_ylabel('Count of Albums')

plt.tight_layout()
plt.show()

print('Imputed-only year distribution:')
for p in [10, 25, 50, 75, 90]:
    print(f'  p{p}: {imputed_gap_clipped.quantile(p/100):.0f}')

### Imputed years skew toward the modern era — an expected bias worth noting

MusicBrainz coverage of pre-1970 releases is sparser, so imputed years for the gap-fill subset
skew later than the true-year distribution. The imputed year is derived from other release dates
in the same release group, so it is not a random fill — but for era classification, imputed albums
should be treated with lower confidence than albums with true dates.

> **Design decision — keep `first_release_year_imputed` as the unified year column in the feature
  block:** it raises coverage significantly. The `first_release_year_imputed` flag column allows
  downstream callers to mask or down-weight imputed values if needed.

---
# Part 5 — Feature Correlations

The raw parquet contains 12 length statistics for each album. Several are algebraically related
(`range = max − min`, `iqr = p75 − p25`, `variance = stddev²`). This section measures pairwise
correlations to confirm which columns are redundant and should be excluded from the model feature
matrix.

In [ ]:
CORR_COLS = [
    'track_count',
    'medium_count',
    'total_length_ms',
    'mean_length_ms',
    'median_length_ms',
    'stddev_length_ms',
    'variance_length_ms',
    'min_length_ms',
    'max_length_ms',
    'range_length_ms',
    'p25_length_ms',
    'p75_length_ms',
    'iqr_length_ms',
    'first_release_year_imputed',
]

sample = df[CORR_COLS].dropna().sample(n=min(50_000, len(df)), random_state=42)
corr = sample.corr(method='spearman')

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, linewidths=0.5, ax=ax, annot_kws={'size': 8}
)
ax.set_title('Spearman Correlation — Track Stats Features (n=50k sample)', weight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Flag pairs with |r| >= 0.95 as redundant
THRESHOLD = 0.95
pairs = []
for i, c1 in enumerate(CORR_COLS):
    for c2 in CORR_COLS[i+1:]:
        r = corr.loc[c1, c2]
        if abs(r) >= THRESHOLD:
            pairs.append((c1, c2, r))

print(f'Pairs with |Spearman r| >= {THRESHOLD}:')
for c1, c2, r in sorted(pairs, key=lambda x: abs(x[2]), reverse=True):
    print(f'  {c1:<30}  {c2:<30}  r={r:+.3f}')

print()
print('Columns excluded from model features (redundant or data-quality flags):')
EXCLUDED = [
    'pct_tracks_with_length',
    'track_count_with_length',
    'variance_length_ms',
    'range_length_ms',
]
for col in EXCLUDED:
    print(f'  {col}')

### Four columns are redundant and excluded from the feature matrix

- `variance_length_ms` is `stddev²` — perfectly correlated with `stddev_length_ms`.
- `range_length_ms` is `max − min` — perfectly correlated with the pair.
- `pct_tracks_with_length` and `track_count_with_length` are data-quality diagnostics, not
  genre signals.

The remaining 12 columns (`first_release_year`, `medium_count`, `track_count`, `total_length_ms`,
`mean_length_ms`, `median_length_ms`, `stddev_length_ms`, `min_length_ms`, `max_length_ms`,
`p25_length_ms`, `p75_length_ms`, `iqr_length_ms`) are kept and passed to MinMaxScaler in
`3-features/06-feature-track-stats.ipynb`.

> **Note on mean vs median:** they are highly correlated for symmetric distributions but diverge
  on albums with a long instrumental outro or a short intro skit. Both are kept because the gap
  between them encodes skew information not captured by any other column.